# Spam vs. Ham

In this project, we will define and create a `spam` vs. `ham` classifier. This classifier will 
be able to take in an email, with its headers and body included, and determine whether or
not the email is `spam`, or `ham`. In this particular notebook, we will undergo the process of
cleaning the email data. Cleaning the email data includes operations such as:

- Processing lines and words into an array of words.
- Processing headers.
- Processing body content.
- Replacing emails, numbers, URLS, punctuation with default values, or empty characters.

> [!NOTE] This project uses data sourced from Apache SpamAssassin's public spam and ham datasets.

## Setup

Before getting started, unzip our data using `./utils/extract.py`. For each of the datasets of 
emails we were given (easy_ham_2, hard_ham, spam_2), we will create a folder with .txt files
containing each email. Below is a sample truncated email 
(`data/easy_ham_2/00002.5a587ae61666c5aa097c8e866aedcc59`):

```bash
'''
From exmh-workers-admin@redhat.com  Wed Aug 21 16:46:03 2002
Return-Path: <exmh-workers-admin@spamassassin.taint.org>
Delivered-To: yyyy@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id 7B74843C32
	for <jm@localhost>; Wed, 21 Aug 2002 11:46:03 -0400 (EDT)
{...truncated, more headers}
\n
Date: Wed, 21 Aug 2002 10:47:32 -0500

--==_Exmh_-2080822444P
Content-Type: text/plain; charset=us-ascii

> From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
> Date:  Wed, 21 Aug 2002 10:40:39 -0500
>
> > From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
{...truncated, more body}
_______________________________________________
Exmh-workers mailing list
Exmh-workers@redhat.com
https://listman.redhat.com/mailman/listinfo/exmh-workers
'''
```

Emails are seen with the following format:
1. **From:** `From <email> <date> <time> <year>`
	* note that there are scenarios where this From field is missing.
2. **Headers:** `<header name>: data`
	* note that if a header is multi-line, it will continue with a newline and a tab (or in some 
    cases, four spaces).
3. **Newline:** there is always a single newline empty line seperating the Headers from the Body.
4. **Body:** any text.

We can also make the following observation: the `From` line is redundant information, the same 
information is seen in the headers always, and the `From` line itself is not always present. 
Therefore we can safely drop this line from being considered.

In [24]:
# use this cell to install all requirements for this project.
# !pip install -r requirements.txt

In [25]:
from bs4 import BeautifulSoup # html parser
from nltk.stem.snowball import SnowballStemmer # word stemmer
from pprint import pprint
import pandas as pd

## Class Creation

### Line Extraction

A great (and simple) first step to begin transforming our email is to simply split each line of a 
file into an array. One step further from this is to create two seperate arrays containing header
and body content. This is easily achievable using the fact that the header and body are ALWAYS 
seperated with a single `\n` character.

In [26]:
DATA_DIR = './data/'
EASY_HAM_2_DIR = DATA_DIR + 'easy_ham_2/'
HARD_HAM_DIR = DATA_DIR + 'hard_ham/'
SPAM_DIR = DATA_DIR + 'spam_2/'

In [27]:
def extract_lines(lines: str):
    data = {
        'headers': [],
        'body': []
    }

    idx = 0
    
    # determine if we have a from line, if so, we skip. 
    if lines[0][:4] == 'From':
        idx += 1
    
    # extract headers
    # the header and body section are split up by a single \n line.
    # once we reach that line, we can move onto the body section.
    while lines[idx] != '\n':
        data['headers'].append(lines[idx])
        idx += 1

    # body
    while idx < len(lines):
        data['body'].append(lines[idx])
        idx += 1

    return data

In [28]:
# easy ham 2 example
eh_ex = dict()
# hard ham example
hh_ex = dict()
# spam example
sp_ex = dict()

examples = (eh_ex, hh_ex, sp_ex)
paths = [
    EASY_HAM_2_DIR + '00003.19be8acd739ad589cd00d8425bac7115',
    HARD_HAM_DIR + '00002.ca96f74042d05c1a1d29ca30467cfcd5',
    SPAM_DIR + '00007.acefeee792b5298f8fee175f9f65c453'
    ]

for i in range(len(examples)):
    with open(paths[i]) as f:
        ex = examples[i]
        ex.update(extract_lines(f.readlines()))

# print('-------------------EASY HAM 2-------------------')
# pprint(eh_ex)

# '-------------------HARD HAM-------------------'
# pprint(hh_ex)

'-------------------SPAM-------------------'
pprint(sp_ex)

{'body': ['\n',
          'NEW PRODUCT ANNOUNCEMENT\n',
          '\n',
          'From: OUTSOURCE ENG.& MFG. INC.\n',
          '\n',
          '\n',
          'Sir/Madam;\n',
          '\n',
          'This note is to inform you of new watchdog board technology for '
          'maintaining\n',
          'continuous unattended operation of PC/Servers etc. that we have '
          'released for\n',
          'distribution.\n',
          '  \n',
          'We are proud to announce Watchdog Control Center featuring MAM '
          '(Multiple\n',
          'Applications Monitor) capability.\n',
          'The key feature of this application enables you to monitor as '
          'many\n',
          'applications as you\n',
          'have resident on any computer as well as the operating system for\n',
          'continuous unattended operation.  The Watchdog Control Center '
          'featuring\n',
          'MAM capability expands third party application "control" of a '
          'Watc

### Header Parsing

Now, with each header, we can parse and extract valuable information. All headers give information
regarding to the emails path, its type, the time it was sent, etc. We can perform the following
transformations to split our headers into readable bits of information:
- Split all words into an array.
- Extract which header a line(s) belongs to.
- Strip all newlines, tabs, commas, chevrons, semicolons (`\t`, `\n`, `,`, `;`, `<`, `>`, `(`, `)`, `[`, `]`).
- Replace all email usernames with `@<domain>`
- Replace all found urls with `URL`

In [29]:
# O(n)
def parse_header_line(line: str):
    if len(line) == 0: return [], False
    # we will need to process a line character by character
    
    # check if line is continuation or new header (is tab or space present?)
    new_header = not (line[0] in {'\t', ' '})

    # single chars to skip over 
    single_char = {',', ';', '<', '>', '\t', '\n', '(', ')', '[', ']'}

    # url to keep track of 
    # we know we have a url when there is http present within the word
    http = ["h", "t", "t", "p"]
    http_idx = 0

    words = []
    curr = "" # current built string

    line += " " # this space ensures we catch the final word
    for c in line:

        # , ; < > 
        if c in single_char:
            continue 

        # USERNAME@<domain>
        if c == "@": # replace username, would be previous word
            curr = "@"
            continue
        
        # URL
        if http_idx == 3:
            pass
        elif c == http[http_idx]:
            http_idx += 1
        else:
            http_idx = 0

        # seperate on space (split)
        if c == " ":
            
            if http_idx == 3: # meaning full http found.
                words.append("URL")
            elif len(curr) > 0: # take care of empty words (double spaces)
                words.append(curr)
            
            curr = ""
            escape = False
            http_idx = 0
            continue

        # add character if passes all these checks.
        curr += c
        
    return words, new_header

# O(n)
def parse_headers(data: dict):
    '''
    Will transform a data object's `headers` key into a dictionary containing each header and its
    entries, numbered by order of appearance.
    '''

    d = dict()

    # sometimes lines continue, this is seen typically by a tab
    # we can also tell by determining that there is no header line present; is there <header name>:?

    curr_header = ''
    curr_val = []
    for line in data['headers']:
        # list of words, is new header?
        l, new_header = parse_header_line(line)
        
        if new_header:
            # save previous values
            if d.get(curr_header):
                d[curr_header].append( curr_val )
            else:
                d[curr_header] = [ curr_val ]

            # rewrite with new values
            curr_header = l[0][:-1] # cut out `:`
            curr_val = l[1:]
        else: # continuation of the previous header
            curr_val += l

    # cleanup
    d.pop('')
    # save previous values
    if d.get(curr_header):
        d[curr_header].append( curr_val )
    else:
        d[curr_header] = [ curr_val ]

    return d

In [30]:
for ex in examples:
    ex['headers'] = parse_headers(ex)

eh_ex, ##hh_ex, sp_ex

({'headers': {'Return-Path': [['@spamassassin.taint.org']],
   'Delivered-To': [['@localhost.netnoteinc.com'],
    ['@listman.spamassassin.taint.org']],
   'Received': [['from',
     'localhost',
     'localhost',
     '127.0.0.1',
     'by',
     'phobos.labs.netnoteinc.com',
     'Postfix',
     'with',
     'ESMTP',
     'id',
     '5134943C34',
     'for',
     '@localhost',
     'Wed',
     '21',
     'Aug',
     '2002',
     '11:18:36',
     '-0400',
     'EDT'],
    ['from',
     'phobos',
     '127.0.0.1',
     'by',
     'localhost',
     'with',
     'IMAP',
     'fetchmail-5.9.0',
     'for',
     '@localhost',
     'single-drop',
     'Wed',
     '21',
     'Aug',
     '2002',
     '16:18:36',
     '+0100',
     'IST'],
    ['from',
     'listman.spamassassin.taint.org',
     'listman.spamassassin.taint.org',
     '66.187.233.211',
     'by',
     'dogma.slashnull.org',
     '8.11.6/8.11.6',
     'with',
     'ESMTP',
     'id',
     'g7LFJuZ30902',
     'for',
     '@jmaso

### Body Parsing

Now, lets format our body. We can perform the following actions on the body:
- Split all words into entries within a list.
- Remove all html formatting. (Anything with an enclosed `<Tag />`; for this operation, we will use
`BeautifulSoup4`, although this may slow our performance, the benefit of having an already-available
tool outweights the costs of developing our own custom tools).
- Remove word endings, (stemming; for this operation, we will use `NLTX` (Natural Language Toolkit), 
for much of the same reasoning as `BeautifulSoup4`).
- Strip all newlines, tabs, and non-alphanumeric characters, allowing (` `) only.
- Replace all email usernames with `@<domain>`.
- Replace all found urls with `URL`.
- Transform all characters into lowercase.
- Replace numerical instances with `NUMBER`.

In [31]:
stemmer = SnowballStemmer('english')

def parse_body_word(word: str):

    # allowed non alnum characters. `@` is included for email checks
    allowed = {' ', '@'} 

    # url to keep track of 
    # we know we have a url when there is http present within the word
    http = ["h", "t", "t", "p"]
    http_idx = 0

    words = []
    curr = ""

    include_period = False # only true if we're currently on an email word.

    word += " " # ensure will run one last time
    for c in word:

        # skip non alphanumeric characters
        if not (c in allowed or c.isalnum()):
            if not (include_period and c == '.'):
                continue

        # USERNAME@<domain>
        if c == "@": # replace username
            curr = "@"
            include_period = True
            continue
        
        # URL
        if http_idx == 3:
            pass
        elif c == http[http_idx]:
            http_idx += 1
        else:
            http_idx = 0

        # seperate
        if c == " ":
            
            if http_idx == 3: # meaning full http found.
                words.append("URL")
            elif len(curr) > 0: # take care of empty words (double spaces)
                w = stemmer.stem(curr)
                
                # alter numbers
                if w.isnumeric(): w = 'NUMBER'

                words.append(w)
            
            curr = ""
            http_idx = 0
            continue

        # add character if passes all these checks. (upper case)
        curr += c
        
    return words

def parse_body(data: dict):
    # first we will combine all lines, then remove all html content. 
    # we must combine all lines within the body to ensure that there is no multi-line html cut off.
    content = ' '.join( data['body'])
    soup = BeautifulSoup(content, 'html.parser')

    text = soup.get_text(separator=' ', strip=True) # converts to one line
    body = text.split(' ') # list of words

    words = []
    for w in body:
        words += parse_body_word(w)

    return words

In [32]:
for ex in examples:
    ex['body'] = parse_body(ex)

sp_ex, hh_ex, eh_ex

({'headers': {'Return-Path': [['@outsrc-em.com']],
   'Delivery-Date': [['Thu', 'Jun', '20', '20:08:33', '2002']],
   'Received': [['from',
     'outsrc-em.com',
     '166.70.149.104',
     'by',
     'dogma.slashnull.org',
     '8.11.6/8.11.6',
     'with',
     'SMTP',
     'id',
     'g5KJ8WI08701',
     'for',
     '@jmason.org',
     'Thu',
     '20',
     'Jun',
     '2002',
     '20:08:32',
     '+0100']],
   'Message-Id': [['@dogma.slashnull.org']],
   'From': [['"Outsource', 'Sales"', '@outsrc-em.com']],
   'To': [['@spamassassin.taint.org"', '@spamassassin.taint.org']],
   'Subject': [['New', 'Product', 'Announcement']],
   'Sender': [['"Outsource', 'Sales"', '@outsrc-em.com']],
   'MIME-Version': [['1.0']],
   'Content-Type': [['text/plain', 'charset="ISO-8859-1"']],
   'Date': [['Fri', '3', 'Jan', '1997', '17:24:47', '-0700']],
   'Reply-To': [['"Outsource', 'Sales"', '@outsrc-em.com']],
   'Content-Transfer-Encoding': [['8bit']],
   'X-Keywords': [[]]},
  'body': ['new',
 

### Extended Parsing

The headers contain a very useful amount of information; yet the question remains: how to properly
process it?

With domain-specific research, we can find that the only mandatory email headers are `From` and 
`Date`. With `To` and `Subject` being common among regular email headers. With `Return-Path`, 
`Message-ID`, and `Recieved` being automatically added by the email server. A detailed 
description of what these headers do can be found [here](https://mailtrap.io/blog/email-headers/).

This greatly reduces the amount of valid information we can potentially recieve from the headers. By
having an uncertain number of headers, we may end up adding unnecessary noise if we were to include
them all. Therefore, it would be more efficient to look at only the most common and most promising
headers.

We will focus on only the following headers, if the header is not present, we will give a `None` 
value. Or, if and only if we are looking for `Received` or `Delivered-To`, we will give it a value 
of `0`.

- *From*: We can extract the email used in the `From` header.
- *Date*: We can seperate the date into different categories like `date_month`, `date_year`, 
`date_day`, `date_time`. We can generalize `date_time` to include the hour of the day, to avoid 
overfitting to specific seconds or minutes of the day.
- *To*: We can extract the email.
- *Subject*: We can perform the same operations as we did on the body, to analyze the text of the 
`Subject` header.
- *Return-Path*: We can extract the email.
- *Received*: We can leave the value as the number of entries within the Recieved path, as it
correlates to how many times an email had gone through different servers to reach a user.
- *Delivered-To*: Similarly to received, we will keep track of the number of times this had
appeared.
- *Message-ID*: We can extract the email.

In [33]:
def extract_headers(data: dict):
    d = dict()

    headers = data.pop('headers')
    # only the mandatory lines we can expect

    # From
    d['from'] = headers['From'][0][-1].lower() # there may be a name, the last element will be an email
    
    # Date:
    # format -- Day DD Mon YYYY HH:MM:SS TZ
    date = headers['Date'][0]
    d['weekday'], d['day'], d['month'], d['year'] = date[0].lower(), date[1], date[2].lower(), date[3]
    d['hour'] = date[4].split(':')[0] # will result in only the hour portion
    d['timezone'] = date[-1] # always last

    # To
    to = headers.get('To')
    if to:
        to = to[0][-1].lower() # the last element is guaranteed to be an email.
    d['to'] = to

    # Subject
    subject = headers.get('Subject')
    if subject:
        words = subject[0]
        subject = []
        for word in words:
            subject += parse_body_word(word)
    d['subject'] = subject

    # Return-Path
    return_path = headers.get('Return-Path')
    if return_path:
        return_path = return_path[0][-1].lower() # extract email 
    d['return_path'] = return_path

    # Recieved
    received = headers.get('Received')
    if received:
        received = len(received)
    else: 
        received = 0
    d['received'] = received

    # Delivered-To
    delivered_to = headers.get('Delivered-To')
    if delivered_to:
        delivered_to = len(delivered_to)
    else:
        delivered_to = 0
    d['delivered_to'] = delivered_to

    # Message-Id
    message_id = headers.get('Message-Id')
    if message_id:
        message_id = message_id[0][-1].lower() # extract email
    d['message_id'] = message_id

    return d

In [34]:
for ex in examples:
    ex.update(extract_headers(ex))

eh_ex #, hh_ex, sp_ex

{'body': ['exmh196335410p',
  'contenttyp',
  'textplain',
  'charsetusascii',
  'from',
  'robert',
  'elz',
  'date',
  'wed',
  'NUMBER',
  'aug',
  'NUMBER',
  'NUMBER',
  'NUMBER',
  'date',
  'tue',
  'NUMBER',
  'aug',
  'NUMBER',
  'NUMBER',
  'NUMBER',
  'from',
  '@vt.edu',
  'messageid',
  '@turingpolice.cc.vt.edu',
  'ever',
  'tri',
  'to',
  'get',
  'mh',
  'to',
  'not',
  'have',
  'a',
  'pseq',
  'sequenc',
  'hmm',
  'ive',
  'been',
  'use',
  'mh',
  'for',
  'a',
  'long',
  'time',
  'sinc',
  'well',
  'befor',
  'there',
  'were',
  'sequenc',
  'and',
  'i',
  'dont',
  'think',
  'ive',
  'ever',
  'seen',
  'a',
  'pseq',
  'im',
  'guess',
  'that',
  'that',
  'the',
  'sequenc',
  'that',
  'you',
  'have',
  'pick',
  'creat',
  'as',
  'i',
  'recal',
  'it',
  'it',
  'has',
  'no',
  'default',
  'sequenc',
  'name',
  'so',
  'the',
  'sequenc',
  'name',
  'that',
  'peopl',
  'use',
  'will',
  'tend',
  'to',
  'vari',
  'from',
  'person',
  'to

### Conversion to DataFrame

Next, to ensure our class can be used in a pipeline, we will define a method to transform our 
processed email into a pandas row.

In [35]:
sample_df = pd.DataFrame([eh_ex, hh_ex, sp_ex])
sample_df.head()

,body,from,weekday,day,month,year,hour,timezone,to,subject,return_path,received,delivered_to,message_id
0,"[exmh196335410p, contenttyp, textplain, charse...",@deepeddy.com,wed,21,aug,2002,10,-0500,@munnari.oz.au,"[re, new, sequenc, window]",@spamassassin.taint.org,13,2,@deepeddy.vircio.com
1,"[may, NUMBER, NUMBER, dear, @arsecandle.org, c...",@mrichi.com,tue,7,may,2002,9,-0600,@arsecandle.org,"[malcolm, in, the, middl, sweepstak, prize, no...",@mrichi.com,6,2,@bocelli.siteprotect.com
2,"[new, product, announc, from, outsourc, eng, m...",@outsrc-em.com,fri,3,jan,1997,17,-0700,@spamassassin.taint.org,"[new, product, announc]",@outsrc-em.com,1,0,@dogma.slashnull.org


## Next Steps

With the previously defined functions, we can now create an `EmailProcessor` class which will 
perform the same set of transformations we had just demonstrated. This class will feature some extra
parameters to alter how exactly we want to parse each email.

Below, we will apply this class to all emails to ensure validity.

In [36]:
from helpers.email_processor import EmailProcessor, extract_filepaths

extractor = EmailProcessor()

ham_filepaths = extract_filepaths('./data/easy_ham_2', './data/hard_ham')
spam_filepaths = extract_filepaths('./data/spam_2')

ham_filepaths, spam_filepaths

(0       data/easy_ham_2/01128.efb36914ecb55d78a894591e...
 1       data/easy_ham_2/00659.02e6dd777f837798533eae8f...
 2       data/easy_ham_2/00776.7df92458e9cf04b8873c406b...
 3       data/easy_ham_2/00116.409b29c26edef06268b4bfa0...
 4       data/easy_ham_2/00615.23556d88fcb1179b25083cfc...
                               ...                        
 1645    data/hard_ham/00022.66e4bce429ab25c5d2c7e8a1a3...
 1646    data/hard_ham/00100.78af3dc4c39277a6e1893f287c...
 1647    data/hard_ham/00241.4e5262894127344225abfc680c...
 1648    data/hard_ham/00110.77ac47048cafa2fe2587182f6d...
 1649    data/hard_ham/00025.c27f4dd57c84c01305bd30bdee...
 Length: 1650, dtype: str,
 0       data/spam_2/01246.d0ee9c7ebf9d953b21de9414cc96...
 1       data/spam_2/01099.f33c6cb5a233f19e1dc1956871c5...
 2       data/spam_2/00346.d93a823fb3350a5da8f0c612ce11...
 3       data/spam_2/00012.cb9c9f2a25196f5b16512338625a...
 4       data/spam_2/00993.32d00ccd0a831830838667fed137...
                             

In [37]:
ham_df = extractor.fit_transform(ham_filepaths)
spam_df = extractor.fit_transform(spam_filepaths)

In [38]:
# label
spam_df['spam'] = True
ham_df['spam'] = False

In [39]:
# display
ham_df

,body,from,weekday,day,month,year,hour,timezone,to,subject,return_path,received,delivered_to,message_id,filename,spam
0,"[on, sun, NUMBER, jul, NUMBER, NUMBER, NUMBER,...",uni.de,sun,21,jul,2002,20,-0400,freshrpms.net,"[re, ximian, apt, repo]",freshrpms.net,7,1,uni.de,01128.efb36914ecb55d78a894591eff0843c5,False
1,"[what, is, mime, mime, stand, for, multipurpos...",docserver.cac.washington.edu,mon,19,aug,2002,23,-0700,example.sourceforge.net,"[wm, the, mime, inform, you, request, last, ch...",example.sourceforge.net,6,1,docserver.cac.washington.edu,00659.02e6dd777f837798533eae8f3b6a0491,False
2,"[im, not, up, to, fork, the, text, but, for, y...",golux.com,tue,13,aug,2002,15,-0400,xent.com,"[a, messag, for, our, time]",xent.com,6,2,golux.com,00776.7df92458e9cf04b8873c406bde7d2fbe,False
3,"[on, sat, jul, NUMBER, NUMBER, at, 121857pm, N...",skynet.ie,sat,20,jul,2002,13,+0100,linux.ie,"[re, ilug, vanquish, the, daemon, of, shell, s...",linux.ie,8,1,skynet.ie,00116.409b29c26edef06268b4bfa03ef1367a,False
4,"[origin, messag, date, thu, NUMBER, aug, NUMBE...",dmv.com,thu,8,aug,2002,16,-0400,example.sourceforge.net,"[re, razorus, use, razor, with, nonmbox, file]",example.sourceforge.net,7,1,landshark,00615.23556d88fcb1179b25083cfc41017f42,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1645,"[tech, updat, today, vital, sign, for, juli, N...",newsletter.online.com,wed,10,jul,2002,06,-0700,spamassassin.taint.org,"[what, to, look, for, in, your, next, smart, p...",newsletter.online.com,2,0,NaN,00022.66e4bce429ab25c5d2c7e8a1a38838a0,False
1646,"[cnet, investor, dispatch, quot, lookup, enter...",newsletter.online.com,wed,17,jul,2002,14,-0700,spamassassin.taint.org,"[newscom, investor, tech, gain, on, intel, mot...",newsletter.online.com,2,0,NaN,00100.78af3dc4c39277a6e1893f287cc2771f,False
1647,"[boundary00an7jyjq1ynhgz79h1wrp, contenttyp, t...",directfreight.com,wed,13,nov,2002,14,-0600,evi-inc.com,"[re, razorus, razorrevok, trust, level, slashd...",example.sourceforge.net,7,1,directfreight.com,00241.4e5262894127344225abfc680c35e3d3,False
1648,"[tech, updat, today, vital, sign, for, juli, N...",newsletter.online.com,thu,18,jul,2002,03,-0700,spamassassin.taint.org,"[when, search, result, dont, count, tech, updat]",newsletter.online.com,2,0,NaN,00110.77ac47048cafa2fe2587182f6d6b19d2,False


In [40]:
spam_df

,body,from,weekday,day,month,year,hour,timezone,to,subject,return_path,received,delivered_to,message_id,filename,spam
0,"[nextpart84815c5abaf209ef376268c8, contenttyp,...",free.fr,fri,2,aug,2002,22,+0200,free.fr,"[dialogu, et, rencontr, rejoin, nous]",free.fr,4,1,free.fr,01246.d0ee9c7ebf9d953b21de9414cc96c2f9,True
1,"[a, man, endow, with, a, NUMBER, hammer, is, s...",myfastmail.com,thu,25,jul,2002,15,-1600,einstein.ssz.com,"[take, your, love, life, to, the, next, level,...",NaN,7,0,NaN,01099.f33c6cb5a233f19e1dc1956871c50681,True
2,"[datemay, NUMBER, hkem.com., dear, sir, i, am,...",hkem.com,fri,17,may,2002,01,-0700,eros-os.org,"[dcmsdev, my, inherit]",eros.cs.jhu.edu,3,0,eros.cs.jhu.edu,00346.d93a823fb3350a5da8f0c612ce1156cd,True
3,"[own, your, veri, own, free, casino, and, spor...",yahoo.com,sat,25,nov,2000,13,-0700,dogma.slashnull.org,"[gain, major, cash]",yahoo.com,2,1,168.191.77.164,00012.cb9c9f2a25196f5b16512338625a85b4,True
4,"[hi, you, can, make, NUMBER, or, more, in, the...",starmail.com,wed,24,jul,0102,04,-0400,locust.minder.net,"[make, NUMBER, in, NUMBER, day, send, email, a...",NaN,7,0,NaN,00993.32d00ccd0a831830838667fed1371d5f,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1391,"[ffffffa9, copyright, NUMBER, all, right, rese...",bol.com.br,mon,05,aug,2002,21,-1700,dogma.slashnull.org,"[best, rate, on, mortgag, in, the, countri]",bol.com.br,4,1,mail.sunwaytech.com.cn,01311.43bfe86df65d53c5f7ca2365dc12582b,True
1392,"[creditfix, thank, you, your, email, address, ...",msn.com,sun,21,jul,2002,10,-1700,netnv.net,"[fix, your, credit, yourself, onlin, NUMBER]",msn.com,6,1,smtp-gw-4.msn.com,00851.dc5452f80ba0bb8481dfc48f70380c4d,True
1393,"[hello, are, you, satisfi, with, your, isp, do...",yahoo.com,wed,07,aug,0102,16,+1000,spamassassin.taint.org,"[new, internet, servic, provid]",xent.com,10,2,ccitih,01328.b23902de23cb3ca1f3334517282372b2,True
1394,"[creditfix, you, can, now, access, and, clear,...",hotmail.com,fri,02,aug,2002,11,-0400,neont.com,"[bad, credit, breakthrough, NUMBER]",hotmail.com,5,1,NaN,01244.9ef966101737a6fc27d8965def288d70,True


In [41]:
ham_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1650 entries, 0 to 1649
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   body          1650 non-null   object
 1   from          1650 non-null   str   
 2   weekday       1650 non-null   str   
 3   day           1650 non-null   str   
 4   month         1650 non-null   str   
 5   year          1650 non-null   str   
 6   hour          1650 non-null   str   
 7   timezone      1645 non-null   str   
 8   to            1639 non-null   str   
 9   subject       1644 non-null   object
 10  return_path   1640 non-null   str   
 11  received      1650 non-null   int64 
 12  delivered_to  1650 non-null   int64 
 13  message_id    1517 non-null   str   
 14  filename      1650 non-null   str   
 15  spam          1650 non-null   bool  
dtypes: bool(1), int64(2), object(2), str(11)
memory usage: 195.1+ KB


In [42]:
spam_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1396 entries, 0 to 1395
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   body          1396 non-null   object
 1   from          1395 non-null   str   
 2   weekday       1396 non-null   str   
 3   day           1396 non-null   str   
 4   month         1396 non-null   str   
 5   year          1396 non-null   str   
 6   hour          1396 non-null   str   
 7   timezone      1255 non-null   str   
 8   to            1350 non-null   str   
 9   subject       1396 non-null   object
 10  return_path   1189 non-null   str   
 11  received      1396 non-null   int64 
 12  delivered_to  1396 non-null   int64 
 13  message_id    1204 non-null   str   
 14  filename      1396 non-null   str   
 15  spam          1396 non-null   bool  
dtypes: bool(1), int64(2), object(2), str(11)
memory usage: 165.1+ KB


In [43]:
email_df = pd.concat([ham_df, spam_df])
email_df = email_df.set_index('filename')
email_df

,body,from,weekday,day,month,year,hour,timezone,to,subject,return_path,received,delivered_to,message_id,spam
filename,,,,,,,,,,,,,,,
01128.efb36914ecb55d78a894591eff0843c5,"[on, sun, NUMBER, jul, NUMBER, NUMBER, NUMBER,...",uni.de,sun,21,jul,2002,20,-0400,freshrpms.net,"[re, ximian, apt, repo]",freshrpms.net,7,1,uni.de,False
00659.02e6dd777f837798533eae8f3b6a0491,"[what, is, mime, mime, stand, for, multipurpos...",docserver.cac.washington.edu,mon,19,aug,2002,23,-0700,example.sourceforge.net,"[wm, the, mime, inform, you, request, last, ch...",example.sourceforge.net,6,1,docserver.cac.washington.edu,False
00776.7df92458e9cf04b8873c406bde7d2fbe,"[im, not, up, to, fork, the, text, but, for, y...",golux.com,tue,13,aug,2002,15,-0400,xent.com,"[a, messag, for, our, time]",xent.com,6,2,golux.com,False
00116.409b29c26edef06268b4bfa03ef1367a,"[on, sat, jul, NUMBER, NUMBER, at, 121857pm, N...",skynet.ie,sat,20,jul,2002,13,+0100,linux.ie,"[re, ilug, vanquish, the, daemon, of, shell, s...",linux.ie,8,1,skynet.ie,False
00615.23556d88fcb1179b25083cfc41017f42,"[origin, messag, date, thu, NUMBER, aug, NUMBE...",dmv.com,thu,8,aug,2002,16,-0400,example.sourceforge.net,"[re, razorus, use, razor, with, nonmbox, file]",example.sourceforge.net,7,1,landshark,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
01311.43bfe86df65d53c5f7ca2365dc12582b,"[ffffffa9, copyright, NUMBER, all, right, rese...",bol.com.br,mon,05,aug,2002,21,-1700,dogma.slashnull.org,"[best, rate, on, mortgag, in, the, countri]",bol.com.br,4,1,mail.sunwaytech.com.cn,True
00851.dc5452f80ba0bb8481dfc48f70380c4d,"[creditfix, thank, you, your, email, address, ...",msn.com,sun,21,jul,2002,10,-1700,netnv.net,"[fix, your, credit, yourself, onlin, NUMBER]",msn.com,6,1,smtp-gw-4.msn.com,True
01328.b23902de23cb3ca1f3334517282372b2,"[hello, are, you, satisfi, with, your, isp, do...",yahoo.com,wed,07,aug,0102,16,+1000,spamassassin.taint.org,"[new, internet, servic, provid]",xent.com,10,2,ccitih,True


In [44]:
# save as csv
email_df.to_csv('./data/processed_emails.csv', index=True)